# Kill mechanism — Norton-Simon vs log-kill

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

The spec's `drug_effect` subsystem (§3) covers kill/inhibition models and names **Norton-Simon**. The Norton-Simon hypothesis: drug-induced regression is proportional to the unperturbed *growth rate*, so a smaller, faster-growing (Gompertz) tumor is more chemo-sensitive — `dV/dt = (g - k·E)·V·ln(Vmax/V)`. This is mechanistically distinct from the log-kill assumption (kill ∝ tumor size) the Claret model uses, so the assumed kill mechanism becomes its own model-selection axis in the divergence view.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.export.registry import get_kernel, kernel_values

ds = onkos.load()
r = ds['drug_effect.norton_simon.nsclc']
print('subsystem:', r.subsystem, '| kernel:', r.kernel)
# Norton-Simon signature: fractional kill rises as the tumor shrinks
spec, v = get_kernel(r), kernel_values(r); v['E'] = 1.0
for size in [40, 80, 160]:
    fk = -spec.rhs(0.0, [size], v)[0] / size
    print(f'  V={size:>3} mm -> fractional kill {fk:.4f}/wk')
assert (-spec.rhs(0.0,[40],v)[0]/40) > (-spec.rhs(0.0,[160],v)[0]/160)

In [ ]:
# Same context, two kill mechanisms: Norton-Simon eradicates; Claret regrows.
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 156, 313)
ns = onkos.simulate(ds, 'drug_effect.norton_simon.nsclc', context=ctx, drug_effect=1.0, t=t)
cl = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0, t=t)
plt.plot(t, ns.tumor_size, label='Norton-Simon'); plt.plot(t, cl.tumor_size, label='Claret log-kill')
plt.xlabel('weeks'); plt.ylabel('tumor (mm)'); plt.legend(); plt.title('kill mechanism shapes the trajectory');
assert ns.tumor_size[-1] < ns.metrics['nadir_tumor_size'] + 1e-6   # no regrowth
assert cl.tumor_size[-1] > cl.metrics['nadir_tumor_size'] * 2       # regrowth

In [ ]:
# Norton-Simon now joins the NSCLC divergence view as a third, distinct model.
cmp = onkos.compare(ds, purpose='tgi', context=ctx, drug_effect=1.0)
print('NSCLC 1L models:', sorted(tr.record_id for tr in cmp.included))
assert 'drug_effect.norton_simon.nsclc' in {tr.record_id for tr in cmp.included}
assert len(cmp.included) >= 3